# A01 · Tensors & the Compute Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a01_tensors.ipynb)

Every model is arithmetic on tensors. Before neurons, learn to read shapes, count FLOPs and bytes, and predict whether an operation is limited by compute or by memory. That habit separates people who use models from people who can design them.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a01). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Read and predict tensor shapes through reshape, transpose, broadcasting and matmul.
- Count the FLOPs of a matrix multiply: $2mkn$.
- Compute memory from shape and dtype, and explain arithmetic intensity.
- Benchmark a matmul on CPU vs GPU and compare with the hardware's peak.

## The math

### Matrix multiply cost

$$C_{ij} = \sum_{p=1}^{k} A_{ip}B_{pj}, \qquad \text{FLOPs} = 2\,m\,k\,n$$

Each of the $mn$ outputs needs $k$ multiplies and $k$ adds.

**Worked example**

1. $X \in \mathbb{R}^{64\times 784},\; W \in \mathbb{R}^{784 \times 10}$ (a batch of 64 flattened MNIST images times a linear layer)
2. $\text{MACs} = 64 \cdot 784 \cdot 10 = 501{,}760$ (multiply-accumulates)
3. $\text{FLOPs} = 2 \cdot 501{,}760 \approx 1.0\times 10^{6}$ (about one megaFLOP: trivial for any GPU)

### Memory and arithmetic intensity

$$\text{bytes} = \text{numel} \times \text{bytes per element}, \qquad I = \frac{\text{FLOPs}}{\text{bytes moved}}$$

A GPU with peak $P$ FLOP/s and bandwidth $B$ bytes/s is compute-bound only when $I > P/B$.

**Worked example**

1. $n=4096$: $A, B, C$ in fp16 $= 3 \cdot 4096^2 \cdot 2 \text{ B} \approx 100.7\text{ MB}$ (square matmul)
2. $\text{FLOPs} = 2 \cdot 4096^3 \approx 1.37\times10^{11}$
3. $I \approx 1.37\times10^{11} / 1.0\times10^{8} \approx 1365$ FLOP/byte (far above an A100's $312\text{T}/2.0\text{T} \approx 156$: compute-bound)
4. Elementwise $y = x + 1$ in fp32: 1 FLOP per 8 bytes (4 read, 4 written) $\Rightarrow I = 0.125$ (memory-bound, which is why kernels get fused)

## Lab: Shapes, broadcasting and a real FLOP/s benchmark

Open the notebook in Colab (Runtime → Change runtime type → T4 GPU) or run locally with `pip install torch`.

### Step 1

Shapes, dtypes, strides and views.

In [ ]:
import torch, time
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(x.shape, x.dtype, x.stride())        # torch.Size([3, 4]) torch.float32 (4, 1)
y = x.T                                     # a view: strides swap, no copy
print(y.stride(), y.is_contiguous())        # (1, 4) False
print(x.element_size() * x.nelement(), "bytes")   # 48

### Step 2

Broadcasting: align from the right, stretch size-1 dims.

In [ ]:
b = torch.tensor([1., 0., -1., 2.])         # (4,)
print((x + b).shape)                        # (3, 4)
col = torch.tensor([[10.], [20.], [30.]])   # (3, 1)
print((x + col).shape)                      # (3, 4)
# einsum states the contraction explicitly
W = torch.randn(4, 5)
print(torch.allclose(x @ W, torch.einsum('ik,kj->ij', x, W)))

### Step 3

Measure achieved FLOP/s for a square matmul and compare with the GPU's peak (T4: ~65 TFLOP/s fp16, ~8 TFLOP/s fp32).

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def bench(n, dtype=torch.float32, iters=10):
    A = torch.randn(n, n, device=device, dtype=dtype)
    B = torch.randn(n, n, device=device, dtype=dtype)
    for _ in range(3): A @ B                       # warm-up
    if device == 'cuda': torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters): A @ B
    if device == 'cuda': torch.cuda.synchronize()   # CUDA is async: wait before stopping the clock
    dt = (time.perf_counter() - t) / iters
    return 2 * n**3 / dt / 1e12
for n in [256, 1024, 4096]:
    print(n, f"{bench(n):.2f} TFLOP/s fp32")
if device == 'cuda':
    print(4096, f"{bench(4096, torch.float16):.2f} TFLOP/s fp16")

### Step 4

Memory-bound vs compute-bound: an elementwise add moves lots of bytes for few FLOPs. Compare its GB/s with the GPU's bandwidth (T4: 320 GB/s).

In [ ]:
n = 50_000_000
a = torch.randn(n, device=device)
for _ in range(3): a + 1
if device == 'cuda': torch.cuda.synchronize()
t = time.perf_counter()
for _ in range(10): b = a + 1
if device == 'cuda': torch.cuda.synchronize()
dt = (time.perf_counter() - t) / 10
print(f"{2 * n * 4 / dt / 1e9:.0f} GB/s moved, {n / dt / 1e12:.4f} TFLOP/s")

### Clean up

Nothing to clean up. In Colab, **Runtime → Disconnect and delete runtime** frees the GPU.

## Check yourself

**1. What does $(32, 1, 128) + (8, 1)$ broadcast to?**

A. $(32, 8, 128)$  
B. $(32, 1, 128)$  
C. An error  
D. $(32, 8, 1)$

<details><summary>Answer</summary>

**A.** Align from the right: $128$ vs $1$ → 128; $1$ vs $8$ → 8; $32$ vs nothing → 32.

</details>

**2. How many FLOPs is $A_{1024\times 4096} \cdot B_{4096\times 1024}$?**

A. $\approx 4.3\times10^{9}$  
B. $\approx 8.6\times10^{9}$  
C. $\approx 1.7\times10^{10}$  
D. $\approx 4.2\times10^{6}$

<details><summary>Answer</summary>

**B.** $2 \cdot 1024 \cdot 4096 \cdot 1024 = 8{,}589{,}934{,}592 \approx 8.6$ GFLOPs.

</details>

**3. LayerNorm, GELU and residual adds take 30% of step time despite few FLOPs. Why?**

A. They use fp64  
B. They are memory-bandwidth bound (low arithmetic intensity)  
C. Autograd is disabled for them  
D. They run on the CPU

<details><summary>Answer</summary>

**B.** Elementwise ops read and write every byte for ~1 FLOP. Kernel fusion (`torch.compile`, fused kernels) cuts the memory traffic.

</details>